# Rutherford scattering

*nuclear · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/rutherford_scattering)

A beam of alpha particles is fired at a gold nucleus. Every alpha is integrated through the repulsive Coulomb field with RK4 in one batched tensor, and the histogram of their scattering angles reproduces Rutherford's $1/\sin^4(\theta/2)$ law — the evidence that the positive charge of an atom sits in a tiny nucleus.

**Vector form**

$$m\,\ddot{\mathbf r} = \frac{k\,\mathbf r}{r^3},\qquad \cot\frac{\theta}{2} = \frac{2Eb}{k},\qquad \frac{d\sigma}{d\Omega} = \Big(\frac{k}{4E}\Big)^2\frac{1}{\sin^4(\theta/2)}$$

**Index (tensor) form**

$$\ddot x_{bi} = \frac{k}{m}\,\frac{x_{bi}}{(x_{bj}x_{bj})^{3/2}},\qquad E_b = \tfrac12 m\,v_{bi}v_{bi} + \frac{k}{r_b},\qquad L_b = m\,\varepsilon_{ij}\,x_{bi}\,v_{bj}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $k$ | `coupling(z_target)` | scalar | $Z_\alpha Z e^2/4\pi\varepsilon_0 = 2Z \cdot 1.44$ MeV fm |
| $x_{bi},\ v_{bi}$ | `y[:, 0], y[:, 1]` | (B, 2) | position (fm) and velocity (units of $c$) of alpha $b$ |
| $x_{bj}x_{bj}$ | `einsum('bj,bj->b', x, x)` | (B,) | repeated $j$ ⇒ summed: squared distance $r_b^2$ |
| $\ddot x_{bi}$ | `acceleration(x, k, m)` | (B, 2) | Coulomb repulsion for the whole beam at once |
| $\varepsilon_{ij}x_{bi}v_{bj}$ | `einsum('ij,...i,...j->...', eps, x, v)` | (B,) | 2D cross product: angular momentum per unit mass |
| $\theta_b$ | `scattering_angle(v)` | (B,) | angle of the outgoing velocity to the beam axis |
| $d\sigma/d\Omega$ | `rutherford_cross_section(theta, E, k)` | (G,) | Rutherford prediction, compared with the histogram |

Units: fm, MeV and time in fm/$c$. Energy and angular momentum are conserved along each path, so $E$ and the impact parameter $b$ alone fix the deflection $\theta = 2\arctan(k/2Eb)$; a head-on alpha turns round at $d = k/E$ (about 45 fm for 5 MeV on gold, still far outside the 7 fm nucleus, so the pure Coulomb law holds). The beam is spread uniformly over a disc of radius $b_{\max}$ (random but with a fixed seed), so the number scattered into a solid angle is (alphas per area) × $d\sigma/d\Omega$. Only one alpha in a few hundred bounces back beyond 90° — Geiger and Marsden's surprise in 1909. Simplifications: the nucleus is a fixed point charge (no recoil, no nuclear force) and the motion is non-relativistic ($v \approx 0.05c$). Each alpha starts 30 $d$ away with exactly the energy and angular momentum of an alpha coming from infinity; the remaining $1/r^2$ tail of the force shifts the angles by well under a degree.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/rutherford_scattering/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import closest_approach, rutherford_angle, rutherford_cross_section, scattering_angle


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    energy, m = params["energy"], tensors.ALPHA_MASS
    k = tensors.coupling(params["z_target"])
    d = k / energy  # head-on distance of closest approach, fm
    start, b_max = 30 * d, params["b_max"] * d

    b_show = tensors.display_impact_parameters(15, min(3.0, params["b_max"]) * d)
    b_beam = tensors.beam_impact_parameters(params["n_beam"], b_max)
    y0 = tensors.initial_state(torch.cat([b_show, b_beam]), energy, k, start, dev)  # (S + B, 2, 2)

    v_inf = math.sqrt(2 * energy / m)
    n_steps = params["n_steps"]
    dt = 2.2 * start / v_inf / n_steps  # long enough for every alpha to fly back out to ~30 d
    paths, y_end, detect = solve.simulate(y0, k, m, dt, n_steps, len(b_show), r_detect=5 * d)

    theta = scattering_angle(y_end[len(b_show) :, 1])  # (B,) final angles of the beam
    detect = detect[len(b_show) :]
    theta_grid = torch.linspace(rutherford_angle(torch.tensor(b_max), energy, k).item(), math.pi, 200)
    flux = params["n_beam"] / (math.pi * b_max**2)  # alphas per fm² of beam
    expected = flux * rutherford_cross_section(theta_grid, energy, k)  # dN/dΩ per sr

    print(f"k = {k:.1f} MeV fm   head-on closest approach d = k/E = {d:.1f} fm")
    for b in [0.5 * d, d, 2 * d]:
        b_t = torch.tensor([b], dtype=torch.float64)
        print(
            f"b = {b:6.1f} fm   θ = {math.degrees(rutherford_angle(b_t, energy, k).item()):6.1f}°"
            f"   r_min = {closest_approach(b_t, energy, k).item():6.1f} fm"
        )
    print(f"fraction scattered beyond 90°: {(theta > math.pi / 2).double().mean().item():.4f}")
    first = int((start - 5.5 * d) / (v_inf * dt))  # the drawn alphas enter the view
    last = min(n_steps, first + int(3 * 5.5 * d / (v_inf * dt)))
    return plot.animate(paths, first, last, theta, detect, theta_grid, expected, d, out_dir / "rutherford.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/rutherford_scattering/equation.py
r"""Physics → tensor translation.

An alpha particle (mass m) moving past a heavy nucleus feels the repulsive Coulomb force

    m \ddot{\mathbf r} = k \, \mathbf r / r^3,        k = Z_α Z e^2 / 4π ε_0 = 2 Z · 1.44 MeV fm.

For a batch of particles b in the scattering plane (components i, j ∈ {x, y}):

    \ddot x_{bi} = (k/m) \, x_{bi} / (x_{bj} x_{bj})^{3/2}          (j summed),

with the conserved energy and angular momentum

    E_b = ½ m v_{bi} v_{bi} + k / r_b,        L_b = m \, ε_{ij} x_{bi} v_{bj}       (i, j summed).

Energy E and impact parameter b fix the scattering angle and the Rutherford cross-section:

    \cot(θ/2) = 2 E b / k,        dσ/dΩ = (k / 4E)^2 / \sin^4(θ/2).
"""

import torch


def acceleration(x: torch.Tensor, k: float, m: float) -> torch.Tensor:
    r"""\ddot x_{bi} = (k/m) x_{bi} / r_b^3,  r_b^2 = x_{bj} x_{bj}.   x: (B, 2) → (B, 2)"""
    r2 = torch.einsum("bj,bj->b", x, x)  # (B,)
    return (k / m) * x / r2[:, None] ** 1.5


def rhs(y: torch.Tensor, k: float, m: float) -> torch.Tensor:
    """dy/dt for y of shape (B, 2, 2): y[:, 0] = position (fm), y[:, 1] = velocity (units of c)."""
    return torch.stack([y[:, 1], acceleration(y[:, 0], k, m)], dim=1)


def energy(y: torch.Tensor, k: float, m: float) -> torch.Tensor:
    r"""E_b = ½ m v_{bi} v_{bi} + k / r_b  (MeV).   (..., 2, 2) → (...)"""
    x, v = y[..., 0, :], y[..., 1, :]
    r = torch.einsum("...i,...i->...", x, x).sqrt()
    return 0.5 * m * torch.einsum("...i,...i->...", v, v) + k / r


def angular_momentum(y: torch.Tensor, m: float) -> torch.Tensor:
    r"""L_b = m ε_{ij} x_{bi} v_{bj}  (MeV fm / c).   (..., 2, 2) → (...)"""
    eps = torch.tensor([[0.0, 1.0], [-1.0, 0.0]], dtype=y.dtype, device=y.device)  # 2D Levi-Civita ε_ij
    return m * torch.einsum("ij,...i,...j->...", eps, y[..., 0, :], y[..., 1, :])


def scattering_angle(v: torch.Tensor) -> torch.Tensor:
    r"""θ_b = angle between the outgoing velocity and the beam axis x.   v: (B, 2) → (B,) in [0, π]"""
    speed = torch.einsum("bi,bi->b", v, v).sqrt()
    return torch.arccos((v[:, 0] / speed).clamp(-1.0, 1.0))


def rutherford_angle(b: torch.Tensor, energy_mev: float, k: float) -> torch.Tensor:
    r"""θ(b) = 2 \arctan(k / 2Eb)."""
    return 2 * torch.atan(k / (2 * energy_mev * b))


def rutherford_cross_section(theta: torch.Tensor, energy_mev: float, k: float) -> torch.Tensor:
    r"""dσ/dΩ = (k/4E)^2 / \sin^4(θ/2)  in fm²/sr."""
    return (k / (4 * energy_mev)) ** 2 / torch.sin(theta / 2) ** 4


def closest_approach(b: torch.Tensor, energy_mev: float, k: float) -> torch.Tensor:
    r"""r_{min} = (d/2)(1 + \sqrt{1 + (2b/d)^2}),  d = k/E the head-on distance."""
    d = k / energy_mev
    return 0.5 * d * (1 + torch.sqrt(1 + (2 * b / d) ** 2))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/rutherford_scattering/tensors.py
"""Build the concrete tensors: coupling constant, impact parameters and the batched initial state."""

import math

import torch

ALPHA_MASS = 3727.379  # MeV / c²
E2 = 1.44  # e² / 4πε0 in MeV fm


def coupling(z_target: int) -> float:
    """k = Z_α Z e²/4πε0 in MeV fm (Z_α = 2)."""
    return 2 * z_target * E2


def display_impact_parameters(n_show: int, b_max: float) -> torch.Tensor:
    """Evenly spaced b from −b_max to +b_max for the trajectories we draw: shape (S,)."""
    return torch.linspace(-b_max, b_max, n_show, dtype=torch.float64)


def beam_impact_parameters(n_beam: int, b_max: float, seed: int = 0) -> torch.Tensor:
    """A uniform beam over a disc of radius b_max: b = b_max √u gives equal flux per unit area: shape (B,)."""
    gen = torch.Generator().manual_seed(seed)
    return b_max * torch.rand(n_beam, generator=gen, dtype=torch.float64).sqrt()


def initial_state(b: torch.Tensor, energy_mev: float, k: float, start: float, device: torch.device) -> torch.Tensor:
    """y0 of shape (B, 2, 2): every alpha starts on the left at distance `start` from the nucleus, moving along +x.

    The speed is set so the total energy is exactly E (part of it is already Coulomb energy k/start), and the
    lateral offset is stretched so the angular momentum is exactly m v∞ b — the beam is then the one that
    came in from infinity with energy E and impact parameter b.
    """
    b = b.to(device)
    v_inf = math.sqrt(2 * energy_mev / ALPHA_MASS)
    v_start = math.sqrt(2 * (energy_mev - k / start) / ALPHA_MASS)
    y = b * v_inf / v_start
    position = torch.stack([-torch.sqrt(start**2 - y**2), y], dim=-1)  # (B, 2), |x| = start
    velocity = torch.stack([torch.full_like(b, v_start), torch.zeros_like(b)], dim=-1)  # (B, 2)
    return torch.stack([position, velocity], dim=1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/rutherford_scattering/solve.py
"""Integrate all alpha particles together with RK4, keeping the full path only for the ones we draw."""

import torch

from problems._common.integrators import rk4_step
from problems._common.progress import steps

from .equation import rhs


def simulate(
    y0: torch.Tensor, k: float, m: float, dt: float, n_steps: int, n_show: int, r_detect: float
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Return (paths (T, S, 2, 2) of the first S = n_show particles, final state (B, 2, 2), detect_step (B,)).

    One RK4 step moves the whole (B, 2, 2) batch. detect_step[b] is the step at which particle b first passed
    r_detect on its way out (n_steps if never) — used to fill the angle histogram in the animation.
    """
    y = y0
    paths = [y[:n_show]]
    detect_step = torch.full((y0.shape[0],), n_steps, dtype=torch.long, device=y0.device)
    for step in steps(n_steps, "RK4 steps"):
        y = rk4_step(lambda s, _t: rhs(s, k, m), y, step * dt, dt)
        paths.append(y[:n_show])
        x, v = y[:, 0], y[:, 1]
        outgoing = (torch.einsum("bi,bi->b", x, x) > r_detect**2) & (torch.einsum("bi,bi->b", x, v) > 0)
        detect_step = torch.where(outgoing & (detect_step == n_steps), step + 1, detect_step)
    return torch.stack(paths), y, detect_step


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/rutherford_scattering/plot.py
"""Left: alpha trajectories past the nucleus. Right: the angle histogram filling up against Rutherford's law."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(
    paths: torch.Tensor,
    first: int,
    last: int,
    theta: torch.Tensor,
    detect_step: torch.Tensor,
    theta_grid: torch.Tensor,
    expected: torch.Tensor,
    d: float,
    path: Path,
) -> Path:
    """paths (T, S, 2, 2); frames run from step `first` to `last`; theta (B,) final angles of the beam;
    expected (G,) Rutherford prediction of dN/dΩ on theta_grid (G,)."""
    pos = paths[:, :, 0, :].cpu().numpy()  # (T, S, 2) in fm
    deg = np.degrees(theta.cpu().numpy())
    detected = detect_step.cpu().numpy()
    view = 5 * d

    fig, (ax_xy, ax_h) = plt.subplots(1, 2, figsize=(9, 4), gridspec_kw={"width_ratios": [1.1, 1]})
    ax_xy.set_xlim(-view, view)
    ax_xy.set_ylim(-0.8 * view, 0.8 * view)
    ax_xy.set_aspect("equal", adjustable="box")
    ax_xy.set_xlabel("x (fm)")
    ax_xy.set_ylabel("y (fm)")
    ax_xy.plot(0, 0, "o", color="#dc2626", ms=7, zorder=5)
    ax_xy.add_patch(plt.Circle((0, 0), d, fill=False, ls="--", color="#9ca3af"))
    ax_xy.text(0.7 * d, -1.0 * d, "d = k/E", fontsize=8, color="#6b7280")
    lines = [ax_xy.plot([], [], lw=1.2, color="#2563eb")[0] for _ in range(pos.shape[1])]
    dots = ax_xy.plot([], [], "o", color="#1e40af", ms=3)[0]

    bins = np.linspace(max(0.0, deg.min()), 180.0, 19)
    solid_angle = 2 * np.pi * (np.cos(np.radians(bins[:-1])) - np.cos(np.radians(bins[1:])))  # sr per bin
    centres = 0.5 * (bins[:-1] + bins[1:])
    ax_h.plot(np.degrees(theta_grid.cpu().numpy()), expected.cpu().numpy(), color="#9ca3af", lw=2, ls="--")
    ax_h.plot([], [], color="#9ca3af", lw=2, ls="--", label=r"Rutherford $\propto 1/\sin^4(\theta/2)$")
    (hist_line,) = ax_h.plot([], [], "o-", color="#2563eb", ms=4, lw=1, label="simulated alphas")
    ax_h.set_yscale("log")
    ax_h.set_xlim(0, 180)
    exp_np = expected.cpu().numpy()
    ax_h.set_ylim(exp_np.min() * 0.2, exp_np.max() * 3)
    ax_h.set_xlabel("scattering angle θ (°)")
    ax_h.set_ylabel("dN/dΩ (per sr)")
    ax_h.legend(loc="upper right", fontsize=8)
    counter = ax_h.text(0.03, 0.04, "", transform=ax_h.transAxes, fontsize=9)

    def update(step: int):
        for s, line in enumerate(lines):
            line.set_data(pos[: step + 1, s, 0], pos[: step + 1, s, 1])
        dots.set_data(pos[step, :, 0], pos[step, :, 1])
        seen = deg[detected <= step]
        counts, _ = np.histogram(seen, bins=bins)
        density = np.where(counts > 0, counts / solid_angle, np.nan)
        hist_line.set_data(centres, density)
        counter.set_text(f"{seen.size} / {deg.size} alphas detected")

    fig.tight_layout()
    frames = [first + i for i in frame_indices(last - first + 1, 60)]
    return save_gif(fig, update, frames, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'energy': 5.0,
    'z_target': 79,
    'b_max': 4.0,
    'n_beam': 2000,
    'n_steps': 3000
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.rutherford_scattering")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.rutherford_scattering.main")
problem_dir = Path("problems/nuclear/rutherford_scattering")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
